# Assistants

[Assistants](https://docs.langchain.com/langsmith/assistants) 为开发者提供了一种快速简便的方式来修改和版本化 agent 以进行实验。

## 向图提供配置

我们的 `task_maistro` 图已经设置为可使用 assistants！

它有一个 `configuration.py` 文件已在图中定义和加载。

我们在图节点中访问可配置字段（`user_id`、`todo_category`、`task_maistro_role`）。

## 创建 assistants

那么，对于我们一直在构建的 `task_maistro` 应用，assistants 的实际用例是什么呢？

对我来说，它能够为不同类别的任务提供独立的待办事项列表。

例如，我想要一个 assistant 用于个人任务，另一个用于工作任务。

使用 `todo_category` 和 `task_maistro_role` 可配置字段可以轻松实现这一点。

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [ ]:
%%capture --no-stderr
%pip install -U langgraph_sdk

这是我们部署图时创建的默认 assistant。

In [ ]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### 个人 assistant

这是我将用于管理个人任务的个人 assistant。

In [ ]:
personal_assistant = await client.assistants.create(
    # "task_maistro" 是我们部署的图的名称
    "task_maistro",
    config={"configurable": {"todo_category": "personal"}}
)
print(personal_assistant)

让我们更新这个 assistant 以包含我的 `user_id` 以方便使用，[为其创建一个新版本](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant)。

In [ ]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

### 工作 assistant

现在，让我们创建一个工作 assistant。我将用它来处理我的工作任务。

In [ ]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

## 使用 assistants

Assistants 将保存到我们部署中的 `Postgres` 中。

这允许我们使用 SDK 轻松 <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [搜索](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) assistants。

In [ ]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

我们可以通过 SDK 轻松管理它们。例如，我们可以删除不再使用的 assistants。
> 视频中的语法略有偏差。以下更新后的代码创建一个备用 assistant 然后将其删除。

In [ ]:
# 创建一个临时 assistant
temp_assistant = await client.assistants.create(
    "task_maistro",
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")

# 删除我们的临时 assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

让我们设置我将使用的 `personal` 和 `work` assistants 的 assistant ID。

In [ ]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### 工作 assistant

让我们为我的工作 assistant 添加一些待办事项。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

In [ ]:
user_input = "Create another ToDo: Finalize set of report generation tutorials."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

assistant 使用其指令在创建任务时进行追问！

它要求我指定一个截止日期 :)

In [ ]:
user_input = "OK, for this task let's get it done by next Tuesday."
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

### 个人 assistant

同样，我们可以为我的个人 assistant 添加待办事项。

In [ ]:
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

In [ ]:
user_input = "Give me a todo summary."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()